In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Setup do ambiente
sys.path.append(str(Path("..").resolve()))
from service.pns_service import get_dataframe, register_derived_variable, list_variables

In [2]:
# -----------------------------------------------------------------------------
# GRUPO 1: DEMOGRAFIA E LOCALIZAÇÃO
# Variáveis Físicas (Silver)
# -----------------------------------------------------------------------------
vars_demo_silver = [
    "sexo",
    "origem",
    "idade",                # Para: faixa_etaria_paper
    "cor_raca",            # Para: dummy_branca
    "estado_civil",        # Para: dummy_casada
    "vive_com_companheiro",# Para: dummy_casada
    "uf",                  # Para: regiao_macro
    "situacao_censitaria"  # Para: dummy_urbano
]

print(">>> Carregando dados físicos (Silver) para inspeção...")
# Filtro base do estudo: Mulheres >= 25 anos
df_silver_demo = get_dataframe(
    variables=vars_demo_silver,
    sources=["2013", "2019"],
    filters={"sexo": "2", "idade": {"operador": ">=", "valor": 25}}
)


>>> Carregando dados físicos (Silver) para inspeção...


In [3]:

# -----------------------------------------------------------------------------
# ANÁLISE EXPLORATÓRIA (Para definição das Regras Gold)
# -----------------------------------------------------------------------------
print(f"\nDataset carregado. Total de linhas: {len(df_silver_demo)}")

cols_to_inspect = [c for c in vars_demo_silver if c != "origem" and c != "idade"]

for col in cols_to_inspect:
    print(f"\n==================================================")
    print(f"VARIÁVEL: {col}")
    print(f"Tipo de dado (dtype): {df_silver_demo[col].dtype}")
    print("Contagem de valores (value_counts):")
    print(df_silver_demo[col].value_counts(dropna=False).head(15))
    
# Para UF, vamos ver uma amostra para confirmar se é código (11) ou sigla (RO)
print(f"\n==================================================")
print(f"VARIÁVEL: uf (Amostra)")
print(df_silver_demo['uf'].unique()[:10])

# Para idade, vamos ver o range
print(f"\n==================================================")
print(f"VARIÁVEL: idade (Range)")
print(f"Min: {df_silver_demo['idade'].min()}, Max: {df_silver_demo['idade'].max()}")



Dataset carregado. Total de linhas: 158910

VARIÁVEL: sexo
Tipo de dado (dtype): object
Contagem de valores (value_counts):
sexo
2    158910
Name: count, dtype: int64

VARIÁVEL: cor_raca
Tipo de dado (dtype): object
Contagem de valores (value_counts):
cor_raca
4    78640
1    62157
2    15779
3     1248
5     1077
9        9
Name: count, dtype: int64

VARIÁVEL: estado_civil
Tipo de dado (dtype): object
Contagem de valores (value_counts):
estado_civil
1    68349
4    43151
5    24383
3    13644
2     9383
Name: count, dtype: int64

VARIÁVEL: vive_com_companheiro
Tipo de dado (dtype): object
Contagem de valores (value_counts):
vive_com_companheiro
sim    56163
1      39013
não    38396
2      25338
Name: count, dtype: int64

VARIÁVEL: uf
Tipo de dado (dtype): object
Contagem de valores (value_counts):
uf
35    12298
31     9475
33     9350
23     7710
21     7074
26     7026
41     7010
15     6859
43     6724
29     6454
13     6128
32     5482
42     5422
25     5395
27     5281
Name:

In [4]:
# -----------------------------------------------------------------------------
# DEFINIÇÃO DE REGRAS GOLD (DEMOGRAFIA)
# -----------------------------------------------------------------------------

# Função auxiliar para lidar com o campo híbrido (sim/1)
def check_sim_hibrido(series):
    # Padroniza: minúsculo, sem espaços, sem .0
    s = series.astype(str).str.replace(r'\.0$', '', regex=True).str.lower().str.strip()
    return s.isin(['1', 'sim', 's'])

# 1. Região Macro
register_derived_variable(
    name="regiao_macro",
    description="Região (Norte, Nordeste...) derivada do primeiro dígito da UF",
    depends_on=["uf"],
    func=lambda df: df["uf"].astype(str).str[0].map({
        '1': 'Norte', '2': 'Nordeste', '3': 'Sudeste', '4': 'Sul', '5': 'Centro-Oeste'
    })
)

# 2. Dummy Branca
register_derived_variable(
    name="dummy_branca",
    description="1 se Branca (código 1), 0 caso contrário",
    depends_on=["cor_raca"],
    func=lambda df: (df["cor_raca"].astype(str).str.replace(r'\.0$', '', regex=True) == '1').astype(int)
)

# 3. Dummy Urbano
register_derived_variable(
    name="dummy_urbano",
    description="1 se Urbano (código 1), 0 caso contrário",
    depends_on=["situacao_censitaria"],
    func=lambda df: (df["situacao_censitaria"].astype(str).str.replace(r'\.0$', '', regex=True) == '1').astype(int)
)

# 4. Dummy Casada (Complexa: envolve campo híbrido)
register_derived_variable(
    name="dummy_casada",
    description="1 se casada (estado_civil=1) OU vive com companheiro (1/sim)",
    depends_on=["estado_civil", "vive_com_companheiro"],
    func=lambda df: (
        (df["estado_civil"].astype(str).str.replace(r'\.0$', '', regex=True) == '1') | 
        check_sim_hibrido(df["vive_com_companheiro"])
    ).astype(int)
)

# 5. Faixa Etária Paper
register_derived_variable(
    name="faixa_etaria_paper",
    description="Faixas etárias do estudo: 25-34, 35-44, 45-54, 55-64, 65+",
    depends_on=["idade"],
    func=lambda df: pd.cut(
        df["idade"], 
        bins=[24, 34, 44, 54, 64, 150], 
        labels=['25-34', '35-44', '45-54', '55-64', '65+']
    )
)

# -----------------------------------------------------------------------------
# VALIDAÇÃO IMEDIATA (GOLD)
# -----------------------------------------------------------------------------
vars_demo_gold = [
    "regiao_macro", "dummy_branca", "dummy_urbano", "dummy_casada", "faixa_etaria_paper"
]

print(">>> Calculando variáveis Gold (Demografia)...")
# Recarregamos pedindo as novas variáveis
df_gold_demo = get_dataframe(
    variables=vars_demo_silver + vars_demo_gold, # Trazemos silver junto para validar
    sources=["2013", "2019"],
    filters={"sexo": "2", "idade": {"operador": ">=", "valor": 25}}
)

print("\n>>> VALIDAÇÃO CRUZADA (Silver vs Gold)")

# Validação: Casada
print("\n--- Validação: Casada ---")
print(pd.crosstab(
    [df_gold_demo['estado_civil'].fillna('NA'), df_gold_demo['vive_com_companheiro'].fillna('NA')], 
    df_gold_demo['dummy_casada']
).head(10)) # Mostra as 10 primeiras combinações para não poluir

# Validação: Região
print("\n--- Validação: Região ---")
print(pd.crosstab(df_gold_demo['uf'].str[0], df_gold_demo['regiao_macro']))

# Validação: Branca
print("\n--- Validação: Branca ---")
print(pd.crosstab(df_gold_demo['cor_raca'], df_gold_demo['dummy_branca']))

>>> Calculando variáveis Gold (Demografia)...

>>> VALIDAÇÃO CRUZADA (Silver vs Gold)

--- Validação: Casada ---
dummy_casada                          0      1
estado_civil vive_com_companheiro             
1            1                        0  26344
             2                        0   1633
             não                      0   2417
             sim                      0  37955
2            1                        0    279
             2                     1449      0
             não                   6138      0
             sim                      0   1517
3            1                        0    790
             2                     2649      0

--- Validação: Região ---
regiao_macro  Centro-Oeste  Nordeste  Norte  Sudeste    Sul
uf                                                         
1                        0         0  31319        0      0
2                        0     53540      0        0      0
3                        0         0      0    36605    

In [5]:
# -----------------------------------------------------------------------------
# GRUPO 2: SOCIOECONÔMICO
# Variáveis Físicas (Silver)
# -----------------------------------------------------------------------------
vars_socio_silver = [
    "escolaridade_nivel",          # Para: anos_estudo_estimado
    "trabalha",                    # Para: dummy_trabalha
    "renda_domiciliar_pc",         # Para: decil_renda
    "tem_filhos_nascidos_vivos"    # Para: dummy_filhos
]

print(">>> Carregando dados físicos (Silver - Socioeconômico) para inspeção...")
df_silver_socio = get_dataframe(
    variables=vars_socio_silver,
    sources=["2013", "2019"],
    filters={"sexo": "2", "idade": {"operador": ">=", "valor": 25}}
)

print(f"\nDataset carregado. Total de linhas: {len(df_silver_socio)}")

for col in vars_socio_silver:
    print(f"\n==================================================")
    print(f"VARIÁVEL: {col}")
    print(f"Tipo de dado (dtype): {df_silver_socio[col].dtype}")
    
    if col == "renda_domiciliar_pc":
        # Para renda, queremos estatísticas, não contagem
        print("Estatísticas Descritivas:")
        print(df_silver_socio[col].describe())

>>> Carregando dados físicos (Silver - Socioeconômico) para inspeção...

Dataset carregado. Total de linhas: 158910

VARIÁVEL: escolaridade_nivel
Tipo de dado (dtype): object

VARIÁVEL: trabalha
Tipo de dado (dtype): object

VARIÁVEL: renda_domiciliar_pc
Tipo de dado (dtype): object
Estatísticas Descritivas:
count       0
unique      0
top       NaN
freq      NaN
Name: renda_domiciliar_pc, dtype: object

VARIÁVEL: tem_filhos_nascidos_vivos
Tipo de dado (dtype): object
